In [ ]:

!git clone https://github.com/tsinghua-fib-lab/AgentSocietyChallenge.git
%cd AgentSocietyChallenge

# 查看 Python 版本及项目文件
!python --version
!ls


In [ ]:

from pathlib import Path

baseline = Path("example/RecAgent_baseline.py")

print("Baseline exists:", baseline.exists())

for keyword in [
    "RecommendationAgent",
    "get_user",
    "get_item",
    "get_reviews",
    "evaluate",
    "InfinigenceLLM"
]:
    print(f"{keyword}:",
          keyword in baseline.read_text())

print("\nRecommendation-related files:")
for p in Path("websocietysimulator").rglob("*.py"):
    if "recommend" in p.name.lower() or "evaluat" in p.name.lower():
        print(p)


In [ ]:

from pathlib import Path

# 确保位于仓库目录
repo = Path("/content/AgentSocietyChallenge")

files = {
    "Baseline Agent": repo / "example/RecAgent_baseline.py",
    "Recommendation Task": repo / "websocietysimulator/tasks/recommendation_task.py",
    "Recommendation Agent": repo / "websocietysimulator/agent/recommendation_agent.py",
}

for name, path in files.items():
    print(f"\n{'='*20} {name} {'='*20}")

    if not path.exists():
        print("File not found:", path)
        continue

    lines = path.read_text(encoding="utf-8").splitlines()

    for i, line in enumerate(lines, start=1):
        if any(keyword in line for keyword in [
            "class ",
            "def ",
            "get_user(",
            "get_item(",
            "get_reviews(",
            "evaluate(",
            "simulator",
            "api_key",
            "model="
        ]):
            print(f"{i}: {line.strip()}")

In [ ]:

from pathlib import Path
import json
import inspect

repo = Path("/content/AgentSocietyChallenge")

# 1. 查看 RecommendationTask 的完整定义
task_file = repo / "websocietysimulator/tasks/recommendation_task.py"

print("=== RECOMMENDATION TASK FORMAT ===")
print(task_file.read_text(encoding="utf-8")[:5000])

# 2. 查看仓库中实际存在的任务文件
print("\n=== SAMPLE TASK FILES ===")

task_files = [
    p for p in repo.rglob("*.json")
    if "task" in str(p).lower()
    and "groundtruth" not in str(p).lower()
]

for p in task_files[:10]:
    print(p.relative_to(repo))

# 3. 查看 Baseline 的数据加载和运行部分
baseline = repo / "example/RecAgent_baseline.py"
lines = baseline.read_text(encoding="utf-8").splitlines()

print("\n=== BASELINE WORKFLOW ===")

for i in range(85, min(185, len(lines))):
    print(f"{i+1}: {lines[i]}")


In [ ]:
{
    "user_id": "...",
    "candidate_category": "...",
    "candidate_list": ["item1", "item2", "..."],
    "loc": [-1, -1]
}

In [ ]:

from pathlib import Path
import json

repo = Path("/content/AgentSocietyChallenge")

# 1. 查找 Amazon 任务和 groundtruth
print("=== AMAZON TASK DIRECTORIES ===")

for p in repo.rglob("amazon"):
    if p.is_dir():
        print(p.relative_to(repo))

# 2. 检查官方示例任务结构
task_files = sorted(
    (repo / "example/track2").glob(
        "*/tasks/*.json"
    )
)

if task_files:
    sample = task_files[0]

    print("\n=== SAMPLE TASK ===")
    print("File:", sample.relative_to(repo))

    data = json.loads(sample.read_text())

    print(json.dumps(
        data,
        indent=2,
        ensure_ascii=False
    )[:2500])

# 3. 查看 Amazon 相关文件
print("\n=== AMAZON FILES ===")

amazon_files = [
    p for p in repo.rglob("*")
    if p.is_file()
    and "amazon" in str(p).lower()
]

for p in amazon_files[:25]:
    print(p.relative_to(repo))

print("\nTotal Amazon-related files:",
      len(amazon_files))


In [ ]:

from pathlib import Path
import json

repo = Path("/content/AgentSocietyChallenge")
amazon = repo / "example/track2/amazon"

tasks = sorted(
    (amazon / "tasks").glob("*.json")
)

groundtruth = sorted(
    (amazon / "groundtruth").glob("*.json")
)

print("Amazon tasks:", len(tasks))
print("Groundtruth files:", len(groundtruth))

print("\n=== SAMPLE GROUNDTRUTH ===")

if groundtruth:
    sample = groundtruth[0]
    print("File:", sample.name)
    print(sample.read_text()[:1500])

print("\n=== TASK USER IDS ===")

task_users = set()

for path in tasks:
    task = json.loads(path.read_text())
    task_users.add(task["user_id"])

print("Unique task users:", len(task_users))

# 与你之前 TVD Pilot 的用户比较
# 如果当前 Notebook 没有 shift_df，
# 可以在原来的 TVD Notebook 中运行匹配部分。

if "shift_df" in globals():
    tvd_users = set(
        shift_df["user_id"].astype(str)
    )

    matched = task_users & tvd_users

    print("TVD users:", len(tvd_users))
    print("Matched users:", len(matched))
else:
    print(
        "shift_df not loaded in this Notebook."
    )


In [ ]:

from google.colab import files
import pandas as pd
import json
from pathlib import Path

uploaded = files.upload()

tvd_df = pd.read_csv(
    "tvd_internal_user_mapping.csv",
    dtype={"user_id": str}
)

repo = Path("/content/AgentSocietyChallenge")
task_dir = repo / "example/track2/amazon/tasks"

task_records = []

for path in task_dir.glob("task_*.json"):
    task = json.loads(path.read_text())

    task_records.append({
        "task_id": path.stem,
        "user_id": str(task["user_id"]),
        "n_candidates": len(task["candidate_list"])
    })

tasks_df = pd.DataFrame(task_records)

matched_df = tasks_df.merge(
    tvd_df,
    on="user_id",
    how="inner"
)

print("Total recommendation tasks:", len(tasks_df))
print("TVD users:", tvd_df["user_id"].nunique())
print("Matched users:", matched_df["user_id"].nunique())
print("Matched tasks:", len(matched_df))

display(matched_df.head())


In [ ]:

from pathlib import Path

repo = Path("/content/AgentSocietyChallenge")

print("=== DATA FILES ===")

extensions = {".csv", ".jsonl", ".parquet", ".db", ".sqlite"}

data_files = [
    p for p in repo.rglob("*")
    if p.is_file()
    and p.suffix.lower() in extensions
]

for p in data_files[:60]:
    print(p.relative_to(repo))

print("\nTotal data files:", len(data_files))

print("\n=== DATA DIRECTORY STRUCTURE ===")

for name in ["example", "websocietysimulator", "data_images"]:
    folder = repo / name
    if folder.exists():
        print(f"\n{name}/")
        for p in list(folder.iterdir())[:20]:
            print("  ", p.name)


In [ ]:

from pathlib import Path

repo = Path("/content/AgentSocietyChallenge")

files_to_check = [
    repo / "websocietysimulator/simulator.py",
    repo / "websocietysimulator/tools/interaction_tool.py",
    repo / "README.md"
]

keywords = [
    "data_dir",
    "download",
    "dataset",
    "get_reviews",
    "review",
    "timestamp",
    "date",
    "huggingface",
    "track2"
]

for path in files_to_check:
    print(f"\n{'='*20} {path.name} {'='*20}")

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    for i, line in enumerate(lines, 1):
        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        ):
            print(f"{i}: {line.strip()[:180]}")


In [ ]:

import requests

repo_id = "SGJQovo/AgentRecBench"

url = (
    f"https://huggingface.co/api/datasets/"
    f"{repo_id}/tree/main"
)

all_files = []
cursor = url

while cursor:
    response = requests.get(
        cursor,
        params={
            "recursive": "true",
            "limit": 1000
        } if cursor == url else None,
        timeout=60
    )
    response.raise_for_status()

    all_files.extend(response.json())

    # Follow Hugging Face pagination if present
    next_url = None
    for part in response.headers.get(
        "Link", ""
    ).split(","):
        if 'rel="next"' in part:
            next_url = part.split(";")[0].strip(" <>")
            break

    cursor = next_url

print("=== AMAZON FILES ===")

amazon_files = [
    f for f in all_files
    if f["type"] == "file"
    and "amazon" in f["path"].lower()
]

for f in amazon_files:
    size_mb = f.get("size", 0) / (1024**2)
    print(
        f"{f['path']} | "
        f"{size_mb:.2f} MB"
    )

print("\nTotal Amazon files:", len(amazon_files))


In [ ]:

import pandas as pd
import requests
from io import StringIO

base = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "task/long_term/amazon/"
)

files_to_check = [
    "amazon.csv",
    "amazon_case.csv",
    "amazon_sample.csv"
]

for filename in files_to_check:
    response = requests.get(
        base + filename,
        timeout=60
    )
    response.raise_for_status()

    df = pd.read_csv(
        StringIO(response.text)
    )

    print(f"\n=== {filename} ===")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if "user_id" in df.columns:
        counts = df.groupby("user_id").size()

        print("Unique users:", len(counts))
        print("Median interactions:", counts.median())
        print("Mean interactions:", round(counts.mean(), 2))
        print("Users with >= 4 interactions:",
              (counts >= 4).sum())
        print("Users with >= 10 interactions:",
              (counts >= 10).sum())

    display(df.head(2))


In [ ]:

# Step 9: Verify the relationship between
# long-term Amazon datasets

full_df = pd.read_csv(base + "amazon.csv")
case_df = pd.read_csv(base + "amazon_case.csv")
sample_df = pd.read_csv(base + "amazon_sample.csv")

keys = ["user_id", "item_id", "date"]

case_users = set(case_df["user_id"])
sample_users = set(sample_df["user_id"])

print("=== USER OVERLAP ===")
print("Case users:", len(case_users))
print("Sample users:", len(sample_users))
print("Overlapping users:", len(case_users & sample_users))

# Identify records in sample but not case,
# accounting for possible duplicate rows
case_counts = (
    case_df.groupby(keys, dropna=False)
    .size()
    .rename("case_n")
)
sample_counts = (
    sample_df.groupby(keys, dropna=False)
    .size()
    .rename("sample_n")
)

comparison = pd.concat(
    [case_counts, sample_counts],
    axis=1
).fillna(0)

comparison["extra_n"] = (
    comparison["sample_n"] - comparison["case_n"]
).clip(lower=0)

extra = comparison[
    comparison["extra_n"] > 0
].reset_index()

print("\n=== SAMPLE-ONLY INTERACTIONS ===")
print("Total extra records:", int(extra["extra_n"].sum()))
print("Users with extra records:", extra["user_id"].nunique())

print("\nExtra records per user:")
print(extra.groupby("user_id")["extra_n"].sum().value_counts())

print("\n=== FULL DATA COVERAGE ===")
print(
    "Case users found in full data:",
    len(case_users & set(full_df["user_id"]))
)

display(extra.head())


In [ ]:

import pandas as pd

# Reconstruct the 100 held-out interactions
extra_targets = extra.loc[
    extra.index.repeat(extra["extra_n"].astype(int)),
    ["user_id", "item_id", "date"]
].copy()

# Latest timestamp in each user's case history
case_last = (
    case_df.groupby("user_id")["date"]
    .max()
    .rename("last_history_date")
)

target_check = extra_targets.merge(
    case_last,
    on="user_id",
    how="left"
)

target_check["is_future"] = (
    target_check["date"] >
    target_check["last_history_date"]
)

print("=== TEMPORAL VALIDATION ===")
print("Total targets:", len(target_check))
print("Targets after all history:",
      target_check["is_future"].sum())
print("Targets not after all history:",
      (~target_check["is_future"]).sum())

print("\n=== EXAMPLES ===")
display(target_check.head())

print("\n=== FULL DATA CHECK ===")

# Check whether each target also appears
# in the full interaction dataset
target_in_full = extra_targets.merge(
    full_df[["user_id", "item_id", "date"]],
    on=["user_id", "item_id", "date"],
    how="left",
    indicator=True
)

print(target_in_full["_merge"].value_counts())


In [ ]:

import requests
from collections import Counter

repo = "SGJQovo/AgentRecBench"
root = (
    f"https://huggingface.co/datasets/{repo}/"
    "resolve/main/task/long_term/amazon/"
)

official_records = []

for i in range(100):
    task_url = root + f"tasks/task_{i}.json"
    gt_url = root + f"groundtruth/groundtruth_{i}.json"

    task_resp = requests.get(task_url, timeout=30)
    gt_resp = requests.get(gt_url, timeout=30)

    task_resp.raise_for_status()
    gt_resp.raise_for_status()

    task = task_resp.json()
    gt = gt_resp.json()

    official_records.append({
        "task_id": i,
        "task": task,
        "groundtruth": gt
    })

print("=== EXAMPLE TASK ===")
print(official_records[0])

print("\n=== TASK FIELDS ===")
print(Counter(
    tuple(sorted(r["task"].keys()))
    for r in official_records
))

print("\n=== GROUNDTRUTH FIELDS ===")
print(Counter(
    tuple(sorted(r["groundtruth"].keys()))
    for r in official_records
))


In [ ]:

# Step 12: Validate all 100 official tasks

target_map = dict(zip(
    extra_targets["user_id"],
    extra_targets["item_id"]
))

results = []

for record in official_records:
    task = record["task"]
    gt = record["groundtruth"]["ground truth"]

    user = task["user_id"]
    candidates = task["candidate_list"]
    heldout = target_map.get(user)

    results.append({
        "task_id": record["task_id"],
        "user_id": user,
        "candidate_count": len(candidates),
        "groundtruth_matches_heldout": gt == heldout,
        "groundtruth_in_candidates": gt in candidates,
        "unique_candidates": len(set(candidates)) == len(candidates),
        "has_case_history": user in case_users
    })

validation_df = pd.DataFrame(results)

print("=== OFFICIAL TASK VALIDATION ===")
print("Tasks:", len(validation_df))
print("Unique users:", validation_df["user_id"].nunique())

for col in [
    "groundtruth_matches_heldout",
    "groundtruth_in_candidates",
    "unique_candidates",
    "has_case_history"
]:
    print(f"{col}: {validation_df[col].sum()}/100")

print("\nCandidate count distribution:")
print(validation_df["candidate_count"].value_counts())

print("\nProblematic tasks:")
display(validation_df[
    ~validation_df[
        [
            "groundtruth_matches_heldout",
            "groundtruth_in_candidates",
            "unique_candidates",
            "has_case_history"
        ]
    ].all(axis=1)
])


In [ ]:

# Step 13: Build expanded chronological holdout dataset

import pandas as pd

df = full_df.copy()

# Sort chronologically within each user
df = df.sort_values(
    ["user_id", "date", "item_id"]
).reset_index(drop=True)

# Assign interaction order
df["interaction_index"] = (
    df.groupby("user_id").cumcount()
)

df["total_interactions"] = (
    df.groupby("user_id")["item_id"].transform("size")
)

# Last interaction becomes target
is_target = (
    df["interaction_index"] ==
    df["total_interactions"] - 1
)

expanded_targets = df[is_target].copy()
expanded_history = df[~is_target].copy()

# Count history interactions
history_counts = (
    expanded_history.groupby("user_id")
    .size()
    .rename("history_length")
)

expanded_targets = expanded_targets.merge(
    history_counts,
    on="user_id",
    how="left"
)

expanded_targets["history_length"] = (
    expanded_targets["history_length"].fillna(0).astype(int)
)

print("=== EXPANDED DATASET ===")
print("Users:", expanded_targets["user_id"].nunique())
print("History interactions:", len(expanded_history))
print("Held-out targets:", len(expanded_targets))

print("\n=== HISTORY LENGTH THRESHOLDS ===")
for threshold in [4, 6, 8, 10]:
    n = (expanded_targets["history_length"] >= threshold).sum()
    print(f"History >= {threshold}: {n} users")

print("\n=== HISTORY LENGTH DISTRIBUTION ===")
display(
    expanded_targets["history_length"]
    .describe()
)

print("\n=== EXAMPLE TARGETS ===")
display(
    expanded_targets[
        ["user_id", "item_id", "date", "history_length"]
    ].head()
)


In [ ]:

# Step 14: Inspect item coverage and overlap

longterm_items = set(full_df["item_id"])
history_items = set(expanded_history["item_id"])
target_items = set(expanded_targets["item_id"])

print("=== LONG-TERM AMAZON ITEMS ===")
print("Unique items (all):", len(longterm_items))
print("Unique history items:", len(history_items))
print("Unique target items:", len(target_items))

print("\n=== HISTORY-TARGET OVERLAP ===")
print(
    "Target items also appearing in history pool:",
    len(target_items & history_items)
)

print("\n=== OFFICIAL CANDIDATE COVERAGE ===")
official_candidates = set()

for record in official_records:
    official_candidates.update(
        record["task"]["candidate_list"]
    )

print("Unique official candidate items:",
      len(official_candidates))
print("Candidates found in long-term interactions:",
      len(official_candidates & longterm_items))

print("\n=== ITEM ID EXAMPLES ===")
print(sorted(list(longterm_items))[:10])


In [ ]:

# Step 15: Search AgentRecBench repository for item metadata

import requests

api_url = (
    "https://huggingface.co/api/datasets/"
    "SGJQovo/AgentRecBench/tree/main"
)

def list_repo_files(path="", recursive=True):
    url = api_url + ("/" + path if path else "")
    response = requests.get(
        url,
        params={
            "recursive": str(recursive).lower(),
            "expand": "false"
        },
        timeout=60
    )
    response.raise_for_status()
    return response.json()

# Use the dataset info endpoint to obtain the full file list
info = requests.get(
    "https://huggingface.co/api/datasets/"
    "SGJQovo/AgentRecBench",
    timeout=60
).json()

all_paths = [
    entry["rfilename"]
    for entry in info.get("siblings", [])
]

keywords = [
    "meta",
    "item.json",
    "product",
    "category",
    "asin",
    "item_info"
]

matches = [
    p for p in all_paths
    if any(k in p.lower() for k in keywords)
]

print("=== METADATA CANDIDATES ===")
print("Total repository files:", len(all_paths))
print("Matching paths:", len(matches))

for path in matches[:100]:
    print(path)


In [ ]:

# Step 16: Inspect metadata files without full downloads

import requests

repo = "SGJQovo/AgentRecBench"

paths = [
    "output_data_long/item.json",
    "output_data_all/item.json",
    "output_data_short/item.json"
]

# File sizes from Hugging Face repository tree
tree_url = (
    f"https://huggingface.co/api/datasets/{repo}/tree/main"
    "?recursive=true"
)

response = requests.get(tree_url, timeout=60)
response.raise_for_status()
tree = response.json()

size_map = {
    f["path"]: f.get("size")
    for f in tree
    if f.get("type") == "file"
}

for path in paths:
    print(f"\n=== {path} ===")

    size = size_map.get(path)
    print(
        "Size (MB):",
        round(size / 1024**2, 2)
        if size is not None else "Unknown"
    )

    url = (
        f"https://huggingface.co/datasets/{repo}"
        f"/resolve/main/{path}"
    )

    # Request only the first 2 KB
    r = requests.get(
        url,
        headers={"Range": "bytes=0-2047"},
        timeout=60,
        stream=True
    )
    r.raise_for_status()

    preview = b""
    for chunk in r.iter_content(chunk_size=2048):
        preview += chunk
        if len(preview) >= 2048:
            break
    r.close()

    print("Preview:")
    print(preview[:800].decode("utf-8", errors="replace"))


In [ ]:

# Step 17: Inspect the end of item.json using HTTP Range

import requests

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

# Check whether server supports byte-range requests
head = requests.head(
    url,
    allow_redirects=True,
    timeout=60
)

print("=== FILE HEADERS ===")
print("Status:", head.status_code)
print("Content-Length:", head.headers.get("Content-Length"))
print("Accept-Ranges:", head.headers.get("Accept-Ranges"))

# Request the final 16 KB
r = requests.get(
    url,
    headers={"Range": "bytes=-16384"},
    stream=True,
    timeout=60
)

print("\n=== RANGE RESPONSE ===")
print("Status:", r.status_code)
print("Content-Range:", r.headers.get("Content-Range"))

if r.status_code == 206:
    content = r.content.decode("utf-8", errors="replace")
    lines = content.splitlines()

    print("\n=== LAST 3 RECORDS ===")
    for line in lines[-3:]:
        print(line[:1000])
        print("---")
else:
    print(
        "Server did not honor the range request. "
        "Skipping download."
    )
    r.close()

In [ ]:

import requests
import json

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

file_size = 1360464643
chunk_size = 16384

for fraction in [0.25, 0.50, 0.75]:
    start = int(file_size * fraction)
    end = start + chunk_size - 1

    response = requests.get(
        url,
        headers={"Range": f"bytes={start}-{end}"},
        timeout=60
    )

    print(f"\n=== POSITION {fraction:.0%} ===")
    print("HTTP status:", response.status_code)

    if response.status_code != 206:
        print("Range request not supported here.")
        continue

    # The first and last lines may be incomplete.
    # Parse only complete JSONL records.
    lines = response.content.split(b"\n")[1:-1]

    parsed = []
    for line in lines:
        try:
            parsed.append(json.loads(line))
        except (json.JSONDecodeError, UnicodeDecodeError):
            continue

    print("Complete records parsed:", len(parsed))

    for item in parsed[:3]:
        print({
            "item_id": item.get("item_id"),
            "source": item.get("source"),
            "title": item.get("title", item.get("name")),
            "categories": item.get("categories"),
            "category": item.get("category")
        })


In [ ]:

# Step 19: Find existing metadata DataFrames in Colab

import pandas as pd

candidates = []

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = [str(c).lower() for c in obj.columns]

        if any(
            keyword in " ".join(cols)
            for keyword in [
                "category",
                "categories",
                "title",
                "parent_asin",
                "asin"
            ]
        ):
            candidates.append({
                "variable": name,
                "rows": len(obj),
                "columns": obj.columns.tolist()[:15]
            })

print("=== POSSIBLE METADATA TABLES ===")

if candidates:
    display(pd.DataFrame(candidates))
else:
    print("No existing metadata DataFrame found.")


In [ ]:

import requests
import json
import pandas as pd
from collections import Counter

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

file_size = 1360464643
chunk_size = 8192

session = requests.Session()
results = []

for pct in range(0, 100, 2):
    start = int(file_size * pct / 100)
    end = min(start + chunk_size - 1, file_size - 1)

    try:
        r = session.get(
            url,
            headers={"Range": f"bytes={start}-{end}"},
            timeout=30
        )
        r.raise_for_status()

        if r.status_code != 206:
            print("Range not honored; stopping.")
            break

        # Discard potentially incomplete boundary lines
        lines = r.content.split(b"\n")[1:-1]
        sources = []

        for line in lines:
            try:
                obj = json.loads(line)
                sources.append(
                    obj.get("source", "unknown")
                )
            except json.JSONDecodeError:
                continue

        most_common = (
            Counter(sources).most_common(1)
        )

        source = (
            most_common[0][0]
            if most_common else "unparsed"
        )

        results.append({
            "position_pct": pct,
            "source": source,
            "parsed_records": len(sources)
        })

    except requests.RequestException as e:
        print(f"Error at {pct}%: {e}")

scan_df = pd.DataFrame(results)

print("=== SOURCE DISTRIBUTION ===")
print(scan_df["source"].value_counts())

print("\n=== NON-GOODREADS REGIONS ===")
display(
    scan_df[
        scan_df["source"] != "goodreads"
    ]
)

print("\n=== AMAZON REGIONS ===")
display(
    scan_df[
        scan_df["source"].astype(str).str.contains(
            "amazon", case=False
        )
    ]
)


In [ ]:

import requests
import json

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

file_size = 1360464643

start = int(file_size * 0.10)
end = start + 65535

r = requests.get(
    url,
    headers={"Range": f"bytes={start}-{end}"},
    timeout=60
)

assert r.status_code == 206, "Range request failed"

# Remove potentially incomplete first and last lines
lines = r.content.split(b"\n")[1:-1]

amazon_items = []

for line in lines:
    try:
        item = json.loads(line)
        if item.get("source") == "amazon":
            amazon_items.append(item)
    except json.JSONDecodeError:
        continue

print("=== AMAZON RECORDS FOUND ===")
print(len(amazon_items))

for item in amazon_items[:3]:
    print("\n=== ITEM ===")
    print("Item ID:", item.get("item_id"))
    print("Title:", item.get("title"))
    print("Categories:", item.get("categories"))
    print("Main category:", item.get("main_category"))
    print("Category:", item.get("category"))
    print("Description:", str(item.get("description"))[:250])
    print("Available fields:", list(item.keys()))


In [ ]:

# Step 22: Extract required Amazon metadata
# using HTTP Range requests

import requests
import json
import pandas as pd

url = (
    "https://huggingface.co/datasets/"
    "SGJQovo/AgentRecBench/resolve/main/"
    "output_data_long/item.json"
)

needed_ids = longterm_items | official_candidates

file_size = 1360464643

# Conservative scan: first 25% of the file
scan_end = int(file_size * 0.25)

chunk_size = 4 * 1024 * 1024
session = requests.Session()

metadata = {}
buffer = b""
position = 0
total_amazon = 0
total_parsed = 0

while position < scan_end:
    end = min(
        position + chunk_size - 1,
        scan_end - 1
    )

    response = session.get(
        url,
        headers={
            "Range": f"bytes={position}-{end}"
        },
        timeout=120
    )
    response.raise_for_status()

    if response.status_code != 206:
        raise RuntimeError(
            "Server did not honor byte-range request"
        )

    data = buffer + response.content
    lines = data.split(b"\n")

    # Preserve incomplete final JSONL record
    buffer = lines.pop()

    for line in lines:
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError:
            continue

        total_parsed += 1

        if item.get("source") != "amazon":
            continue

        total_amazon += 1
        item_id = str(item.get("item_id", ""))

        if item_id in needed_ids:
            metadata[item_id] = {
                "item_id": item_id,
                "main_category": item.get(
                    "main_category"
                ),
                "categories": item.get(
                    "categories"
                ),
                "title": item.get("title"),
                "description": item.get(
                    "description"
                )
            }

    position = end + 1

    if position % (40 * 1024 * 1024) < chunk_size:
        print(
            f"Scanned {position / 1024**2:.0f} MB | "
            f"Matched {len(metadata)} items"
        )

# Parse the final buffered record if complete
if buffer.strip():
    try:
        item = json.loads(buffer)
        if item.get("source") == "amazon":
            total_amazon += 1
            item_id = str(item.get("item_id", ""))
            if item_id in needed_ids:
                metadata[item_id] = {
                    "item_id": item_id,
                    "main_category": item.get(
                        "main_category"
                    ),
                    "categories": item.get(
                        "categories"
                    ),
                    "title": item.get("title"),
                    "description": item.get(
                        "description"
                    )
                }
    except json.JSONDecodeError:
        pass

metadata_df = pd.DataFrame(
    list(metadata.values())
)

print("\n=== EXTRACTION RESULTS ===")
print("Parsed records:", total_parsed)
print("Amazon records:", total_amazon)
print("Needed unique items:", len(needed_ids))
print("Matched items:", len(metadata_df))

print("\n=== COVERAGE ===")
print(
    "Long-term item coverage:",
    f"{len(longterm_items & set(metadata)) / len(longterm_items):.2%}"
)

print(
    "Official candidate coverage:",
    f"{len(official_candidates & set(metadata)) / len(official_candidates):.2%}"
)

display(metadata_df.head())

# Save locally in Colab
metadata_df.to_csv(
    "/content/amazon_metadata_matched.csv",
    index=False
)

print("\nSaved: /content/amazon_metadata_matched.csv")


In [ ]:

# Step 23: Inspect category quality and coverage

import pandas as pd
import ast
from collections import Counter

meta = metadata_df.copy()

def parse_categories(value):
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return parsed
        except (ValueError, SyntaxError):
            pass
    return []

meta["category_path"] = meta["categories"].apply(
    parse_categories
)

meta["level_1"] = meta["category_path"].apply(
    lambda x: x[0] if len(x) >= 1 else None
)

meta["level_2"] = meta["category_path"].apply(
    lambda x: x[1] if len(x) >= 2 else None
)

print("=== METADATA QUALITY ===")
print("Total items:", len(meta))

for col in ["main_category", "level_1", "level_2"]:
    valid = meta[col].notna().sum()
    print(
        f"{col}: {valid}/{len(meta)} "
        f"({valid / len(meta):.2%})"
    )
    print("Unique categories:", meta[col].nunique())

print("\n=== TOP LEVEL-1 CATEGORIES ===")
display(meta["level_1"].value_counts().head(15))

print("\n=== TOP LEVEL-2 CATEGORIES ===")
display(meta["level_2"].value_counts().head(20))

# Match category metadata to user history
history_with_cat = expanded_history.merge(
    meta[["item_id", "level_1", "level_2"]],
    on="item_id",
    how="left",
    validate="many_to_one"
)

print("\n=== HISTORY COVERAGE ===")
print("History interactions:", len(history_with_cat))

for col in ["level_1", "level_2"]:
    valid = history_with_cat[col].notna().sum()
    print(
        f"{col}: {valid}/{len(history_with_cat)} "
        f"({valid / len(history_with_cat):.2%})"
    )

# How many users have at least 3 valid category
# observations in both chronological halves?
history_with_cat = history_with_cat.sort_values(
    ["user_id", "date", "item_id"]
).copy()

history_with_cat["position"] = (
    history_with_cat.groupby("user_id").cumcount()
)

history_with_cat["n_history"] = (
    history_with_cat.groupby("user_id")["item_id"]
    .transform("size")
)

history_with_cat["period"] = (
    history_with_cat["position"] <
    history_with_cat["n_history"] // 2
).map({True: "early", False: "recent"})

for col in ["level_1", "level_2"]:
    counts = (
        history_with_cat
        .dropna(subset=[col])
        .groupby(["user_id", "period"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            columns=["early", "recent"],
            fill_value=0
        )
    )

    eligible = (
        (counts["early"] >= 3) &
        (counts["recent"] >= 3)
    ).sum()

    print(
        f"\nUsers with >=3 valid {col} "
        f"observations in BOTH halves: {eligible}"
    )

In [ ]:

# Step 24: Compute user-level Raw and Excess TVD

import numpy as np
import pandas as pd
from collections import Counter

B = 500
rng = np.random.default_rng(2026)

def calculate_tvd(a, b):
    count_a = Counter(a)
    count_b = Counter(b)

    categories = set(count_a) | set(count_b)

    return 0.5 * sum(
        abs(
            count_a[c] / len(a)
            - count_b[c] / len(b)
        )
        for c in categories
    )

def measure_shift(values, B=500):
    values = np.asarray(values, dtype=object)

    n = len(values)
    split = n // 2

    early = values[:split]
    recent = values[split:]

    observed = calculate_tvd(early, recent)

    shuffled = np.empty(B)

    for b in range(B):
        perm = rng.permutation(values)
        shuffled[b] = calculate_tvd(
            perm[:split],
            perm[split:]
        )

    null_mean = shuffled.mean()
    excess = observed - null_mean

    p_value = (
        1 + np.sum(shuffled >= observed)
    ) / (B + 1)

    return observed, null_mean, excess, p_value

records = []

for user_id, group in history_with_cat.groupby(
    "user_id", sort=False
):
    group = group.sort_values(
        ["date", "item_id"]
    )

    row = {
        "user_id": user_id,
        "history_length": len(group)
    }

    for level in ["level_1", "level_2"]:
        valid_values = (
            group[level].dropna().tolist()
        )

        if len(valid_values) < 6:
            row[f"{level}_raw_tvd"] = np.nan
            row[f"{level}_null_mean"] = np.nan
            row[f"{level}_excess_tvd"] = np.nan
            row[f"{level}_pvalue"] = np.nan
            continue

        raw, null, excess, p = measure_shift(
            valid_values, B=B
        )

        row[f"{level}_raw_tvd"] = raw
        row[f"{level}_null_mean"] = null
        row[f"{level}_excess_tvd"] = excess
        row[f"{level}_pvalue"] = p

    records.append(row)

shift_df = pd.DataFrame(records)

print("=== MEASUREMENT SAMPLE ===")
print(
    "Users with Level-2 TVD:",
    shift_df["level_2_raw_tvd"].notna().sum()
)

print("\n=== LEVEL-2 RESULTS ===")
display(
    shift_df[
        [
            "level_2_raw_tvd",
            "level_2_null_mean",
            "level_2_excess_tvd",
            "level_2_pvalue"
        ]
    ].describe()
)

print("\n=== LEVEL-1 RESULTS ===")
display(
    shift_df[
        [
            "level_1_raw_tvd",
            "level_1_null_mean",
            "level_1_excess_tvd"
        ]
    ].describe()
)

print("\n=== CORRELATION ===")
print(
    shift_df[
        [
            "level_1_raw_tvd",
            "level_2_raw_tvd",
            "level_2_excess_tvd",
            "history_length"
        ]
    ].corr().round(3)
)

shift_df.to_csv(
    "/content/amazon_user_preference_shift.csv",
    index=False
)

print("\nSaved: /content/amazon_user_preference_shift.csv")


In [ ]:

# Step 25: Strict chronological TVD + bootstrap CI

import numpy as np
import pandas as pd
from collections import Counter

B = 500
rng = np.random.default_rng(2026)

def tvd(a, b):
    ca, cb = Counter(a), Counter(b)
    keys = set(ca) | set(cb)
    return 0.5 * sum(
        abs(ca[k] / len(a) - cb[k] / len(b))
        for k in keys
    )

results = []

for user_id, group in history_with_cat.groupby("user_id"):
    group = group.sort_values(
        ["date", "item_id"]
    )

    n = len(group)
    split = n // 2

    early = group.iloc[:split]["level_2"].dropna().tolist()
    recent = group.iloc[split:]["level_2"].dropna().tolist()

    if len(early) < 3 or len(recent) < 3:
        continue

    observed = tvd(early, recent)

    values = np.array(early + recent, dtype=object)
    n_early = len(early)

    null_values = np.empty(B)

    for b in range(B):
        shuffled = rng.permutation(values)
        null_values[b] = tvd(
            shuffled[:n_early],
            shuffled[n_early:]
        )

    null_mean = null_values.mean()

    results.append({
        "user_id": user_id,
        "history_length": n,
        "early_valid": len(early),
        "recent_valid": len(recent),
        "raw_tvd": observed,
        "null_mean": null_mean,
        "excess_tvd": observed - null_mean,
        "permutation_p": (
            1 + np.sum(null_values >= observed)
        ) / (B + 1)
    })

measurement_df = pd.DataFrame(results)

# User-level bootstrap for mean excess TVD
values = measurement_df["excess_tvd"].to_numpy()
rng_boot = np.random.default_rng(2027)

boot_means = np.array([
    rng_boot.choice(
        values, size=len(values), replace=True
    ).mean()
    for _ in range(5000)
])

ci_low, ci_high = np.quantile(
    boot_means, [0.025, 0.975]
)

print("=== STRICT MEASUREMENT RESULTS ===")
print("Users:", len(measurement_df))
print("Mean Raw TVD:", round(measurement_df["raw_tvd"].mean(), 4))
print("Mean Null TVD:", round(measurement_df["null_mean"].mean(), 4))
print("Mean Excess TVD:", round(values.mean(), 4))
print("95% Bootstrap CI:",
      (round(ci_low, 4), round(ci_high, 4)))

print("\n=== USER-LEVEL SIGNAL ===")
print(
    "Users with positive Excess TVD:",
    (measurement_df["excess_tvd"] > 1e-10).sum()
)
print(
    "Users with permutation p < 0.05:",
    (measurement_df["permutation_p"] < 0.05).sum()
)

measurement_df.to_csv(
    "/content/amazon_tvd_final.csv",
    index=False
)

print("\nSaved: /content/amazon_tvd_final.csv")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

# Use strict chronological measurement results
df = measurement_df.copy()

fig, axes = plt.subplots(
    1, 3, figsize=(15, 4.3),
    gridspec_kw={"width_ratios": [1.2, 1.2, 1]}
)

blue = "#3478A5"
gray = "#A0A7B0"
orange = "#D17A35"

# Panel A: Observed vs shuffle-null distributions
ax = axes[0]
bins = np.linspace(0, 1, 26)

ax.hist(
    df["raw_tvd"],
    bins=bins,
    density=True,
    alpha=0.55,
    color=blue,
    label="Observed TVD"
)
ax.hist(
    df["null_mean"],
    bins=bins,
    density=True,
    alpha=0.45,
    color=gray,
    label="Mean shuffle-null TVD"
)

ax.set_title("(A) Observed vs. Shuffle Null")
ax.set_xlabel("Total Variation Distance")
ax.set_ylabel("Density")
ax.legend(frameon=False, fontsize=8)

# Panel B: Excess TVD distribution
ax = axes[1]
excess = df["excess_tvd"].to_numpy()

ax.hist(
    excess, bins=35,
    color=blue, alpha=0.8,
    edgecolor="white", linewidth=0.4
)
ax.axvline(
    0, color="black",
    linestyle="--", linewidth=1.2,
    label="No excess shift"
)
ax.axvline(
    excess.mean(),
    color=orange,
    linewidth=2,
    label=f"Mean = {excess.mean():.3f}"
)

ax.set_title("(B) Excess Preference Shift")
ax.set_xlabel("Excess TVD")
ax.set_ylabel("Number of users")
ax.legend(frameon=False, fontsize=8)

# Panel C: Mean Excess TVD + bootstrap CI
ax = axes[2]

rng = np.random.default_rng(2027)
boot_means = np.array([
    rng.choice(
        excess,
        size=len(excess),
        replace=True
    ).mean()
    for _ in range(5000)
])

ci_low, ci_high = np.quantile(
    boot_means, [0.025, 0.975]
)

mean = excess.mean()

ax.errorbar(
    mean, 0,
    xerr=[
        [mean - ci_low],
        [ci_high - mean]
    ],
    fmt="o",
    markersize=9,
    capsize=6,
    linewidth=2.5,
    color=blue
)

ax.axvline(
    0, color="black",
    linestyle="--", linewidth=1.2
)

ax.set_xlim(-0.015, 0.065)
ax.set_ylim(-0.7, 0.7)
ax.set_yticks([])
ax.set_title("(C) Population Mean Excess TVD")
ax.set_xlabel("Mean Excess TVD")

ax.text(
    mean, 0.22,
    f"{mean:.4f}\n95% CI [{ci_low:.4f}, {ci_high:.4f}]",
    ha="center", va="bottom",
    fontsize=10
)

# Shared styling
for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=9)

fig.suptitle(
    "Preference Shifts Beyond Random Variation",
    fontsize=15, fontweight="bold", y=1.04
)

fig.text(
    0.5, -0.04,
    f"Amazon Long-term | n = {len(df):,} users | "
    "Level-2 product categories | 500 permutations per user",
    ha="center", fontsize=10, color="dimgray"
)

plt.tight_layout()

plt.savefig(
    "/content/figure1_preference_shift.png",
    dpi=300, bbox_inches="tight"
)
plt.savefig(
    "/content/figure1_preference_shift.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:

import os
import glob
import json

# Find official Amazon recommendation tasks
task_files = glob.glob(
    "/content/**/amazon/tasks/task_0.json",
    recursive=True
)

print("=== FOUND TASK FILES ===")
print(task_files)

if not task_files:
    print("\nTask 0 not found. Checking available directories:")
    print("AgentSocietyChallenge exists:",
          os.path.exists("/content/AgentSocietyChallenge"))
else:
    task_path = next(
        (p for p in task_files
         if "AgentSocietyChallenge" in p),
        task_files[0]
    )

    # Find matching groundtruth file
    amazon_dir = os.path.dirname(
        os.path.dirname(task_path)
    )
    gt_files = glob.glob(
        os.path.join(
            amazon_dir, "**", "groundtruth_0.json"
        ),
        recursive=True
    )

    print("\n=== TASK 0 ===")
    with open(task_path, "r") as f:
        task = json.load(f)
    print(task)

    print("\n=== GROUNDTRUTH ===")
    print("Found:", gt_files)
    if gt_files:
        with open(gt_files[0], "r") as f:
            gt = json.load(f)
        print(gt)

print("\n=== AVAILABLE DATAFRAMES ===")
for name in [
    "case_df",
    "sample_df",
    "full_df",
    "metadata_df",
    "measurement_df"
]:
    print(
        f"{name}:",
        "available" if name in globals() else "missing"
    )


In [ ]:

import os
import json

base = (
    "/content/AgentSocietyChallenge/"
    "example/track2/amazon"
)

task_path = os.path.join(
    base, "tasks", "task_0.json"
)

gt_path = os.path.join(
    base, "groundtruth", "groundtruth_0.json"
)

with open(task_path, "r") as f:
    task = json.load(f)

with open(gt_path, "r") as f:
    gt = json.load(f)

print("=== OFFICIAL TRACK 2 TASK ===")
print("Task type:", task.get("type"))
print("User ID:", task.get("user_id"))
print("Candidate category:", task.get("candidate_category"))
print("Number of candidates:",
      len(task.get("candidate_list", [])))
print("Candidates:", task.get("candidate_list"))

print("\n=== GROUNDTRUTH ===")
print(gt)

print("\n=== VALIDATION ===")
assert task["type"] == "recommendation"
assert len(task["candidate_list"]) == 20
assert gt["ground truth"] in task["candidate_list"]

print("Correct Track 2 recommendation task!")


In [ ]:

# Step 28: Construct official recommendation prompt

import pandas as pd
import json
import ast

user_id = task["user_id"]
candidate_ids = [str(x) for x in task["candidate_list"]]

# Normalize ID types
history = case_df.copy()
history["user_id"] = history["user_id"].astype(str)
history["item_id"] = history["item_id"].astype(str)

history = history[
    history["user_id"] == str(user_id)
].sort_values(["date", "item_id"])

meta_lookup = metadata_df.copy()
meta_lookup["item_id"] = meta_lookup["item_id"].astype(str)
meta_lookup = (
    meta_lookup.drop_duplicates("item_id")
    .set_index("item_id")
)

def format_item(item_id):
    if item_id not in meta_lookup.index:
        return f"{item_id}: Metadata unavailable"

    row = meta_lookup.loc[item_id]
    title = str(row.get("title", ""))[:130]

    categories = row.get("categories", [])
    if isinstance(categories, str):
        try:
            categories = ast.literal_eval(categories)
        except (ValueError, SyntaxError):
            categories = []

    if not isinstance(categories, list):
        categories = []

    category = " > ".join(
        str(x) for x in categories[:3]
    )

    return (
        f"{item_id} | {title} | "
        f"Category: {category}"
    )

history_lines = [
    f"{i+1}. {format_item(item_id)}"
    for i, item_id in enumerate(history["item_id"])
]

candidate_lines = [
    f"{i+1}. {format_item(item_id)}"
    for i, item_id in enumerate(candidate_ids)
]

recommendation_prompt = f"""
You are a product recommendation assistant.

TASK:
Rank all 20 candidate products according to how likely
the user is to choose them next.

USER PURCHASE HISTORY:
Listed from oldest to most recent.

{chr(10).join(history_lines)}

CANDIDATE PRODUCTS:
{chr(10).join(candidate_lines)}

INSTRUCTIONS:
- Infer the user's preferences from purchase history.
- Pay attention to recent preferences.
- Rank all 20 candidates from most to least likely.
- Use only the provided information.
- Return only a JSON object with one key, "ranking".
- The ranking must contain exactly 20 unique product IDs.
- Do not include explanations.
"""

print("=== USER HISTORY ===")
print("User ID:", user_id)
print("Number of interactions:", len(history))
print("\n".join(history_lines[:10]))

print("\n=== CANDIDATE METADATA COVERAGE ===")
print(
    sum(x in meta_lookup.index for x in candidate_ids),
    "/ 20"
)

print("\n=== PROMPT SIZE ===")
print("Characters:", len(recommendation_prompt))
print("Estimated tokens:", len(recommendation_prompt) // 4)

print("\n=== PROMPT PREVIEW ===")
print(recommendation_prompt[:2200])

# Check that groundtruth is not explicitly
# supplied as a labeled answer
assert "ground truth" not in recommendation_prompt.lower()


In [ ]:

# Step 29: Verify the correct recommendation dataset

import pandas as pd

eligible = set(
    measurement_df["user_id"].astype(str)
)

history_users = set(
    expanded_history["user_id"].astype(str)
)

target_users = set(
    expanded_targets["user_id"].astype(str)
)

metadata_items = set(
    metadata_df["item_id"].astype(str)
)

history_items = set(
    expanded_history["item_id"].astype(str)
)

target_items = set(
    expanded_targets["item_id"].astype(str)
)

print("=== USER COVERAGE ===")
print("Measurement users:", len(eligible))
print(
    "Users with history:",
    len(eligible & history_users)
)
print(
    "Users with target:",
    len(eligible & target_users)
)

print("\n=== ITEM METADATA COVERAGE ===")
print(
    "History item coverage:",
    f"{len(history_items & metadata_items) / len(history_items):.2%}"
)
print(
    "Target item coverage:",
    f"{len(target_items & metadata_items) / len(target_items):.2%}"
)

print("\n=== HISTORY LENGTH ===")
hist_lengths = (
    expanded_history[
        expanded_history["user_id"].astype(str).isin(eligible)
    ]
    .groupby("user_id")
    .size()
)

print(hist_lengths.describe())

print("\n=== VALIDATION ===")
assert eligible <= history_users
assert eligible <= target_users
assert history_items <= metadata_items
assert target_items <= metadata_items

print("Ready to construct recommendation tasks!")


In [ ]:

import pandas as pd
import ast

# Metadata with category hierarchy
meta = metadata_df.copy()

def parse_cat(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            value = ast.literal_eval(x)
            return value if isinstance(value, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

meta["category_path"] = meta["categories"].apply(parse_cat)
meta["level_1"] = meta["category_path"].apply(
    lambda x: x[0] if len(x) >= 1 else None
)
meta["level_2"] = meta["category_path"].apply(
    lambda x: x[1] if len(x) >= 2 else None
)

eligible = set(measurement_df["user_id"].astype(str))

targets = expanded_targets.copy()
targets["user_id"] = targets["user_id"].astype(str)
targets["item_id"] = targets["item_id"].astype(str)
targets = targets[
    targets["user_id"].isin(eligible)
].merge(
    meta[["item_id", "level_1", "level_2"]],
    on="item_id",
    how="left",
    validate="many_to_one"
)

level2_counts = meta.groupby(
    ["level_1", "level_2"]
)["item_id"].nunique()

level1_counts = meta.groupby(
    "level_1"
)["item_id"].nunique()

targets["same_l2_pool"] = targets.apply(
    lambda r: level2_counts.get(
        (r["level_1"], r["level_2"]), 0
    ) - 1
    if pd.notna(r["level_2"]) else 0,
    axis=1
)

targets["same_l1_other_l2_pool"] = targets.apply(
    lambda r: (
        level1_counts.get(r["level_1"], 0)
        - level2_counts.get(
            (r["level_1"], r["level_2"]), 0
        )
    ) if pd.notna(r["level_1"]) else 0,
    axis=1
)

print("=== CANDIDATE POOL CHECK ===")
print("Eligible targets:", len(targets))

print(
    "Targets with >=9 same-L2 negatives:",
    (targets["same_l2_pool"] >= 9).sum()
)

print(
    "Targets with >=10 same-L1 other-L2 negatives:",
    (targets["same_l1_other_l2_pool"] >= 10).sum()
)

print(
    "Targets satisfying both:",
    (
        (targets["same_l2_pool"] >= 9)
        &
        (targets["same_l1_other_l2_pool"] >= 10)
    ).sum()
)

print("\n=== POOL SIZE SUMMARY ===")
display(
    targets[
        [
            "same_l2_pool",
            "same_l1_other_l2_pool"
        ]
    ].describe()
)

print("\n=== MISSING TARGET CATEGORIES ===")
print(targets["level_2"].isna().sum())


In [ ]:

# Step 31: Generate reproducible 20-item candidate sets

import numpy as np
import pandas as pd
import json

rng = np.random.default_rng(2026)

meta_pool = meta.drop_duplicates("item_id").copy()
meta_pool["item_id"] = meta_pool["item_id"].astype(str)

all_items = set(meta_pool["item_id"])

l1_pools = {
    k: set(g["item_id"])
    for k, g in meta_pool.dropna(subset=["level_1"])
    .groupby("level_1")
}

l2_pools = {
    k: set(g["item_id"])
    for k, g in meta_pool.dropna(
        subset=["level_1", "level_2"]
    ).groupby(["level_1", "level_2"])
}

history_map = (
    expanded_history
    .assign(
        user_id=lambda x: x["user_id"].astype(str),
        item_id=lambda x: x["item_id"].astype(str)
    )
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

tasks = []

for _, row in targets.iterrows():
    user = str(row["user_id"])
    target = str(row["item_id"])

    l1 = row["level_1"]
    l2 = row["level_2"]

    excluded = history_map.get(user, set()) | {target}

    same_l2 = (
        l2_pools.get((l1, l2), set()) - excluded
        if pd.notna(l1) and pd.notna(l2)
        else set()
    )

    same_l1 = (
        l1_pools.get(l1, set()) - excluded - same_l2
        if pd.notna(l1)
        else set()
    )

    selected = []

    def sample_from(pool, n):
        available = sorted(set(pool) - set(selected))
        k = min(n, len(available))
        if k == 0:
            return []
        return rng.choice(
            available, size=k, replace=False
        ).tolist()

    selected += sample_from(same_l2, 9)
    selected += sample_from(same_l1, 10)

    # Fill remaining positions from the same L1 first
    if len(selected) < 19 and pd.notna(l1):
        selected += sample_from(
            l1_pools.get(l1, set()) - excluded,
            19 - len(selected)
        )

    # Global fallback
    if len(selected) < 19:
        selected += sample_from(
            all_items - excluded,
            19 - len(selected)
        )

    assert len(selected) == 19

    candidates = selected + [target]
    rng.shuffle(candidates)

    tasks.append({
        "user_id": user,
        "groundtruth": target,
        "candidate_list": candidates,
        "target_level_1": l1,
        "target_level_2": l2,
        "repeat_target": target in history_map.get(user, set()),
        "same_l2_negatives": sum(
            x in same_l2 for x in selected
        )
    })

recommendation_tasks = pd.DataFrame(tasks)

print("=== GENERATED TASKS ===")
print("Number of tasks:", len(recommendation_tasks))
print(
    "All have 20 unique candidates:",
    recommendation_tasks["candidate_list"].apply(
        lambda x: len(x) == len(set(x)) == 20
    ).all()
)
print(
    "All include groundtruth:",
    recommendation_tasks.apply(
        lambda r: r["groundtruth"] in r["candidate_list"],
        axis=1
    ).all()
)

print("\n=== CANDIDATE SAMPLING ===")
print(
    recommendation_tasks[
        "same_l2_negatives"
    ].describe()
)

print("\n=== REPEAT TARGETS ===")
print(
    recommendation_tasks[
        "repeat_target"
    ].value_counts()
)

print("\n=== EXAMPLE TASK ===")
display(recommendation_tasks.head(1))

recommendation_tasks.to_json(
    "/content/amazon_recommendation_tasks.json",
    orient="records",
    indent=2
)

print("\nSaved: /content/amazon_recommendation_tasks.json")


In [ ]:

# Step 32: Build the first recommendation prompt

import pandas as pd
import json
import ast

example = recommendation_tasks.iloc[0]

user_id = str(example["user_id"])
groundtruth = str(example["groundtruth"])
candidate_ids = list(example["candidate_list"])

history = expanded_history.copy()
history["user_id"] = history["user_id"].astype(str)
history["item_id"] = history["item_id"].astype(str)

user_history = (
    history[history["user_id"] == user_id]
    .sort_values(["date", "item_id"])
)

meta_lookup = (
    metadata_df
    .assign(item_id=lambda x: x["item_id"].astype(str))
    .drop_duplicates("item_id")
    .set_index("item_id")
)

def format_product(item_id):
    row = meta_lookup.loc[item_id]

    title = str(row["title"])[:140]
    categories = row["categories"]

    if isinstance(categories, str):
        try:
            categories = ast.literal_eval(categories)
        except (ValueError, SyntaxError):
            categories = []

    if not isinstance(categories, list):
        categories = []

    category = " > ".join(
        str(c) for c in categories[:3]
    )

    return (
        f"{item_id} | {title} | "
        f"Category: {category}"
    )

history_lines = [
    f"{i+1}. {format_product(item)}"
    for i, item in enumerate(user_history["item_id"])
]

candidate_lines = [
    f"{i+1}. {format_product(item)}"
    for i, item in enumerate(candidate_ids)
]

recommendation_prompt = f"""
You are an Amazon product recommendation assistant.

TASK:
Rank the 20 candidate products according to how likely
the user is to select them next.

USER HISTORY (oldest to most recent):
{chr(10).join(history_lines)}

CANDIDATE PRODUCTS:
{chr(10).join(candidate_lines)}

INSTRUCTIONS:
- Infer user preferences from the purchase history.
- Consider both long-term and recent preferences.
- Rank all 20 candidates from most to least likely.
- Use only the provided information.
- Return valid JSON in the following format:
  {{"ranking": ["item_id_1", "item_id_2", "..."]}}
- Include exactly 20 unique candidate IDs.
- Do not provide explanations.
"""

print("=== FIRST RECOMMENDATION TASK ===")
print("User ID:", user_id)
print("History interactions:", len(user_history))
print("Candidate products:", len(candidate_ids))
print("Metadata coverage:",
      sum(i in meta_lookup.index for i in candidate_ids),
      "/ 20")

print("\n=== PROMPT SIZE ===")
print("Characters:", len(recommendation_prompt))
print("Estimated tokens:", len(recommendation_prompt) // 4)

print("\n=== HISTORY PREVIEW ===")
print("\n".join(history_lines[:5]))

print("\n=== CANDIDATE PREVIEW ===")
print("\n".join(candidate_lines[:5]))

assert len(user_history) >= 6
assert all(i in meta_lookup.index for i in candidate_ids)
assert groundtruth in candidate_ids

print("\nPrompt ready for LLM!")


In [ ]:

# Step 33A: List available Gemini models

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("=== AVAILABLE GEMINI MODELS ===")

available_models = []

for model in client.models.list():
    name = model.name

    if "gemini" in name.lower():
        available_models.append(name)

        print(name)

print("\nTotal Gemini models:", len(available_models))

print("\n=== RECOMMENDED MODEL CHECK ===")

for name in available_models:
    if "flash" in name.lower():
        print(name)


In [ ]:

# Step 33C — Robust Gemini single-task pilot

import json
import time
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

models_to_try = [
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-3.8-flash"
]

response = None
used_model = None

for model_name in models_to_try:
    print(f"\nTrying: {model_name}")

    for attempt in range(2):
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=recommendation_prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0
                )
            )

            if not response.text:
                raise ValueError("Empty model response")

            result = json.loads(response.text)
            ranking = result["ranking"]

            valid = (
                isinstance(ranking, list)
                and len(ranking) == 20
                and len(set(ranking)) == 20
                and set(ranking) == set(candidate_ids)
            )

            if not valid:
                raise ValueError(
                    "Invalid candidate ranking"
                )

            used_model = model_name
            break

        except APIError as e:
            print(
                f"Attempt {attempt + 1}: "
                f"API error {e.code}"
            )

            # Only retry transient errors
            if e.code not in (429, 500, 502, 503, 504):
                break

            if attempt == 0:
                time.sleep(5)

        except (ValueError, KeyError, TypeError) as e:
            print(
                f"Attempt {attempt + 1}: "
                f"{type(e).__name__}: {e}"
            )
            break

    if used_model is not None:
        break

if used_model is None:
    print("\nNo valid ranking obtained.")
    print("No recommendation result was recorded.")

else:
    target_rank = ranking.index(groundtruth) + 1
    hit_at_5 = int(target_rank <= 5)

    print("\n=== SUCCESS ===")
    print("Model:", used_model)
    print("Valid ranking:", True)

    print("\n=== TOP 5 ===")
    for i, item in enumerate(ranking[:5], 1):
        print(f"{i}. {item}")

    print("\n=== EVALUATION ===")
    print("Groundtruth:", groundtruth)
    print("Groundtruth rank:", target_rank)
    print("Hit@5:", hit_at_5)
    print("Failure@5:", 1 - hit_at_5)

    print("\n=== TOKEN USAGE ===")
    print(response.usage_metadata)


In [ ]:

# Step 34 — 10-user Gemini recommendation pilot

import json
import time
import pandas as pd
from pathlib import Path
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata

MODEL = "gemini-3.1-flash-lite"
OUTPUT = Path("/content/gemini_pilot_10.json")

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

# Use a fixed, reproducible sample.
# Exclude the first user already evaluated.
pilot_tasks = (
    recommendation_tasks.iloc[1:]
    .sample(n=10, random_state=2026)
)

# Reuse the formatting function from Step 32.
def build_prompt(row):
    uid = str(row["user_id"])
    candidates = list(row["candidate_list"])

    user_hist = (
        history[history["user_id"] == uid]
        .sort_values(["date", "item_id"])
    )

    history_text = "\n".join(
        f"{i+1}. {format_product(item)}"
        for i, item in enumerate(user_hist["item_id"])
    )

    candidate_text = "\n".join(
        f"{i+1}. {format_product(item)}"
        for i, item in enumerate(candidates)
    )

    prompt = f"""
You are an Amazon product recommendation assistant.

TASK:
Rank the 20 candidate products according to how likely
the user is to select them next.

USER HISTORY (oldest to most recent):
{history_text}

CANDIDATE PRODUCTS:
{candidate_text}

INSTRUCTIONS:
- Infer user preferences from the purchase history.
- Consider both long-term and recent preferences.
- Rank all 20 candidates from most to least likely.
- Use only the provided information.
- Return valid JSON in the following format:
  {{"ranking": ["item_id_1", "item_id_2", "..."]}}
- Include exactly 20 unique candidate IDs.
- Do not provide explanations.
"""
    return prompt

# Resume if this cell has already been partially run
if OUTPUT.exists():
    results = json.loads(OUTPUT.read_text())
else:
    results = []

completed = {r["user_id"] for r in results}

for i, (_, row) in enumerate(pilot_tasks.iterrows(), 1):
    uid = str(row["user_id"])

    if uid in completed:
        continue

    candidates = list(row["candidate_list"])
    target = str(row["groundtruth"])
    prompt = build_prompt(row)

    record = {
        "user_id": uid,
        "model": MODEL,
        "groundtruth": target,
        "valid": False,
        "ranking": None,
        "target_rank": None,
        "hit_at_5": None,
        "failure_at_5": None,
        "input_tokens": None,
        "output_tokens": None,
        "error": None
    }

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                temperature=0
            )
        )

        ranking = json.loads(response.text)["ranking"]

        valid = (
            isinstance(ranking, list)
            and len(ranking) == 20
            and len(set(ranking)) == 20
            and set(ranking) == set(candidates)
        )

        record["valid"] = valid
        record["ranking"] = ranking

        if valid:
            rank = ranking.index(target) + 1
            record["target_rank"] = rank
            record["hit_at_5"] = int(rank <= 5)
            record["failure_at_5"] = int(rank > 5)
        else:
            record["error"] = "Invalid ranking"

        usage = response.usage_metadata
        if usage:
            record["input_tokens"] = usage.prompt_token_count
            record["output_tokens"] = usage.candidates_token_count

    except (APIError, ValueError, KeyError, TypeError) as e:
        record["error"] = f"{type(e).__name__}: {str(e)[:250]}"

    results.append(record)
    OUTPUT.write_text(
        json.dumps(results, indent=2, ensure_ascii=False)
    )

    print(
        f"[{i}/10] "
        f"Valid={record['valid']} "
        f"Hit@5={record['hit_at_5']} "
        f"Error={record['error']}"
    )

    time.sleep(2)

# Summary
df_pilot = pd.DataFrame(results)
valid_df = df_pilot[df_pilot["valid"]]

print("\n=== PILOT SUMMARY ===")
print("Total attempted:", len(df_pilot))
print("Valid rankings:", len(valid_df))
print("Validity rate:", round(df_pilot["valid"].mean(), 3))

if len(valid_df):
    print("Hit@5:", round(valid_df["hit_at_5"].mean(), 3))
    print("Failure@5:", round(valid_df["failure_at_5"].mean(), 3))
    print("Mean input tokens:", valid_df["input_tokens"].mean())
    print("Mean output tokens:", valid_df["output_tokens"].mean())

print("\nSaved:", OUTPUT)


In [ ]:

# Step 35 — Recover failed pilot tasks

import json
import time
import random
import pandas as pd
from pathlib import Path
from google.genai import types
from google.genai.errors import APIError

OUTPUT = Path("/content/gemini_pilot_10.json")
results = json.loads(OUTPUT.read_text())

# Match users to their original recommendation tasks
pilot_lookup = {
    str(row["user_id"]): row
    for _, row in pilot_tasks.iterrows()
}

failed = [r for r in results if not r["valid"]]

print("Tasks requiring retry:", len(failed))

for record in failed:
    uid = record["user_id"]
    row = pilot_lookup[uid]

    candidates = list(row["candidate_list"])
    target = str(row["groundtruth"])
    prompt = build_prompt(row)

    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite",
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0
                )
            )

            ranking = json.loads(response.text)["ranking"]

            valid = (
                isinstance(ranking, list)
                and len(ranking) == 20
                and len(set(ranking)) == 20
                and set(ranking) == set(candidates)
            )

            if not valid:
                record["error"] = "Invalid ranking"
                print("Invalid ranking:", uid)
                break

            rank = ranking.index(target) + 1
            usage = response.usage_metadata

            record.update({
                "valid": True,
                "ranking": ranking,
                "target_rank": rank,
                "hit_at_5": int(rank <= 5),
                "failure_at_5": int(rank > 5),
                "input_tokens": (
                    usage.prompt_token_count if usage else None
                ),
                "output_tokens": (
                    usage.candidates_token_count if usage else None
                ),
                "error": None
            })

            print(
                f"Recovered: {uid} | "
                f"Rank={rank} | Hit@5={int(rank <= 5)}"
            )
            break

        except APIError as e:
            record["error"] = (
                f"{type(e).__name__}: {str(e)[:250]}"
            )

            if e.code not in (429, 500, 502, 503, 504):
                print("Non-retryable error:", e.code)
                break

            if attempt < 3:
                wait = min(60, 5 * 2**attempt)
                wait += random.uniform(0, 2)
                print(
                    f"Attempt {attempt + 1}: "
                    f"API {e.code}, retry in {wait:.1f}s"
                )
                time.sleep(wait)

        except (ValueError, KeyError, TypeError) as e:
            record["error"] = str(e)
            print("Parsing error:", e)
            break

    # Save after each task
    OUTPUT.write_text(
        json.dumps(results, indent=2, ensure_ascii=False)
    )

# Updated results
df_pilot = pd.DataFrame(results)
valid_df = df_pilot[df_pilot["valid"]]

print("\n=== UPDATED PILOT SUMMARY ===")
print("Total tasks:", len(df_pilot))
print("Valid tasks:", len(valid_df))
print("Remaining failures:", (~df_pilot["valid"]).sum())

if len(valid_df):
    print("Hit@5:", round(valid_df["hit_at_5"].mean(), 3))
    print("Failure@5:", round(valid_df["failure_at_5"].mean(), 3))

print("\nSaved:", OUTPUT)


In [ ]:

# Step 36 — Audit and freeze the recommendation prompt

import hashlib
import json
from pathlib import Path

MODEL = "gemini-3.1-flash-lite"
PROMPT_VERSION = "v1.0"

# Inspect one task using the existing Step 34 function
audit_row = recommendation_tasks.iloc[0]
audit_prompt = build_prompt(audit_row)

audit_candidates = list(audit_row["candidate_list"])
audit_target = str(audit_row["groundtruth"])

print("=== EXPERIMENT CONFIGURATION ===")
print("Model:", MODEL)
print("Prompt version:", PROMPT_VERSION)
print("Temperature:", 0)
print("Candidates per task:", len(audit_candidates))

print("\n=== PROMPT VALIDATION ===")
print("Prompt length:", len(audit_prompt))
print("Unique candidates:", len(set(audit_candidates)))
print("Groundtruth in candidates:",
      audit_target in audit_candidates)

# Groundtruth must appear as a candidate,
# but must NOT be explicitly identified as the answer.
print("Explicit groundtruth label in prompt:",
      "groundtruth" in audit_prompt.lower())

print("TVD mentioned in prompt:",
      "tvd" in audit_prompt.lower())

print("\n=== PROMPT PREVIEW ===")
print(audit_prompt[:1800])

# Record configuration for reproducibility
config = {
    "model": MODEL,
    "prompt_version": PROMPT_VERSION,
    "temperature": 0,
    "candidate_count": 20,
    "top_k": 5,
    "prompt_template_function": "build_prompt",
    "prompt_sha256_example": hashlib.sha256(
        audit_prompt.encode("utf-8")
    ).hexdigest(),
    "candidate_sampling_seed": 2026,
    "dataset": "AgentRecBench Amazon Long-term",
    "evaluation": "custom benchmark-derived candidate ranking"
}

config_path = Path("/content/recommendation_experiment_config.json")

config_path.write_text(
    json.dumps(config, indent=2, ensure_ascii=False)
)

print("\n=== CONFIG SAVED ===")
print(config_path)


In [ ]:

# Step 37 — Run 50-user recommendation batch

import json
import time
import random
import hashlib
import pandas as pd
from pathlib import Path
from google.genai import types
from google.genai.errors import APIError

MODEL = "gemini-3.1-flash-lite"
OUTPUT = Path("/content/gemini_batch_50.json")

# Exclude all 10 pilot users
pilot_user_ids = {
    str(r["user_id"])
    for r in json.loads(
        Path("/content/gemini_pilot_10.json").read_text()
    )
}

eligible_tasks = recommendation_tasks[
    ~recommendation_tasks["user_id"].astype(str).isin(
        pilot_user_ids
    )
]

batch_tasks = eligible_tasks.sample(
    n=50,
    random_state=2026
)

# Resume from previous results if available
if OUTPUT.exists():
    results = json.loads(OUTPUT.read_text())
else:
    results = []

# Preserve the selected task set for reproducibility
task_ids = set(batch_tasks["user_id"].astype(str))
results = [
    r for r in results
    if str(r["user_id"]) in task_ids
]

result_lookup = {
    str(r["user_id"]): r
    for r in results
}

for i, (_, row) in enumerate(batch_tasks.iterrows(), 1):
    uid = str(row["user_id"])

    # Skip already valid results
    if uid in result_lookup and result_lookup[uid]["valid"]:
        continue

    candidates = list(row["candidate_list"])
    target = str(row["groundtruth"])
    prompt = build_prompt(row)

    record = {
        "user_id": uid,
        "model": MODEL,
        "prompt_version": "v1.0",
        "prompt_sha256": hashlib.sha256(
            prompt.encode("utf-8")
        ).hexdigest(),
        "groundtruth": target,
        "valid": False,
        "ranking": None,
        "target_rank": None,
        "hit_at_5": None,
        "failure_at_5": None,
        "input_tokens": None,
        "output_tokens": None,
        "error": None
    }

    for attempt in range(4):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0
                )
            )

            ranking = json.loads(response.text)["ranking"]

            valid = (
                isinstance(ranking, list)
                and len(ranking) == 20
                and len(set(ranking)) == 20
                and set(ranking) == set(candidates)
            )

            if not valid:
                record["error"] = "Invalid ranking"
                break

            rank = ranking.index(target) + 1
            usage = response.usage_metadata

            record.update({
                "valid": True,
                "ranking": ranking,
                "target_rank": rank,
                "hit_at_5": int(rank <= 5),
                "failure_at_5": int(rank > 5),
                "input_tokens": (
                    usage.prompt_token_count if usage else None
                ),
                "output_tokens": (
                    usage.candidates_token_count if usage else None
                ),
                "error": None
            })
            break

        except APIError as e:
            record["error"] = (
                f"{type(e).__name__}: {str(e)[:250]}"
            )

            if e.code not in (429, 500, 502, 503, 504):
                break

            if attempt < 3:
                wait = min(60, 5 * 2**attempt)
                wait += random.uniform(0, 2)
                print(
                    f"[{i}/50] API {e.code}, "
                    f"retry in {wait:.1f}s"
                )
                time.sleep(wait)

        except (ValueError, KeyError, TypeError) as e:
            record["error"] = str(e)
            break

    result_lookup[uid] = record
    results = list(result_lookup.values())

    # Save after every user
    OUTPUT.write_text(
        json.dumps(results, indent=2, ensure_ascii=False)
    )

    print(
        f"[{i}/50] "
        f"Valid={record['valid']} | "
        f"Hit@5={record['hit_at_5']} | "
        f"Error={record['error']}"
    )

    time.sleep(2)

# Summary
df_batch = pd.DataFrame(results)
valid_df = df_batch[df_batch["valid"]]

print("\n=== BATCH 50 SUMMARY ===")
print("Attempted:", len(df_batch))
print("Valid:", len(valid_df))
print("Invalid:", len(df_batch) - len(valid_df))

if len(valid_df):
    print("Hit@5:", round(valid_df["hit_at_5"].mean(), 3))
    print("Failure@5:", round(valid_df["failure_at_5"].mean(), 3))
    print("Mean input tokens:", round(
        valid_df["input_tokens"].mean(), 1
    ))
    print("Mean output tokens:", round(
        valid_df["output_tokens"].mean(), 1
    ))

print("\nSaved:", OUTPUT)


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the actual TVD measurement results
df = measurement_df.copy()

required = ["raw_tvd", "null_mean", "excess_tvd"]
assert all(c in df.columns for c in required), "Missing TVD columns"

df = df.dropna(subset=required)

# Summary statistics from actual data
raw_mean = df["raw_tvd"].mean()
null_mean = df["null_mean"].mean()
excess_mean = df["excess_tvd"].mean()

fig, axes = plt.subplots(
    1, 2, figsize=(10, 3.8),
    gridspec_kw={"width_ratios": [1.25, 1]}
)

# Panel A: distributions
axes[0].hist(
    df["raw_tvd"],
    bins=np.linspace(0, 1, 21),
    alpha=0.65,
    label="Observed TVD",
    color="#477FA6",
    edgecolor="white"
)

axes[0].hist(
    df["excess_tvd"],
    bins=np.linspace(-1, 1, 41),
    alpha=0.55,
    label="Excess TVD",
    color="#D5A06D",
    edgecolor="white"
)

axes[0].axvline(
    0, color="gray", linestyle="--", linewidth=1
)

axes[0].set_title(
    "A. Preference-shift distributions",
    fontsize=11, fontweight="bold"
)
axes[0].set_xlabel("TVD value")
axes[0].set_ylabel("Number of users")
axes[0].legend(frameon=False, fontsize=9)
axes[0].spines[["top", "right"]].set_visible(False)

# Panel B: observed vs permutation null
bars = axes[1].bar(
    ["Observed TVD", "Shuffled TVD"],
    [raw_mean, null_mean],
    color=["#477FA6", "#AAB8C5"],
    width=0.55
)

for bar, value in zip(bars, [raw_mean, null_mean]):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.012,
        f"{value:.3f}",
        ha="center", fontsize=10
    )

axes[1].set_ylim(0, 0.65)
axes[1].set_ylabel("Mean TVD")
axes[1].set_title(
    "B. Permutation validity check",
    fontsize=11, fontweight="bold"
)
axes[1].spines[["top", "right"]].set_visible(False)

axes[1].text(
    0.5, 0.90,
    f"Mean Excess TVD = {excess_mean:.3f}",
    transform=axes[1].transAxes,
    ha="center", fontsize=10,
    fontweight="bold"
)

fig.suptitle(
    f"Preference Shift and Permutation Validation (n = {len(df):,})",
    fontsize=13, fontweight="bold", y=1.04
)

plt.tight_layout()

# Save high-resolution and vector versions
plt.savefig(
    "/content/Figure2_TVD_Validation.png",
    dpi=300, bbox_inches="tight"
)
plt.savefig(
    "/content/Figure2_TVD_Validation.pdf",
    bbox_inches="tight"
)

plt.show()

print("=== ACTUAL MEASUREMENT RESULTS ===")
print("Users:", len(df))
print("Mean Raw TVD:", round(raw_mean, 4))
print("Mean Null TVD:", round(null_mean, 4))
print("Mean Excess TVD:", round(excess_mean, 4))
print("\nSaved Figure2_TVD_Validation.png and .pdf")


In [ ]:

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the actual pilot results
path = "/content/gemini_pilot_10.json"

with open(path, "r") as f:
    data = json.load(f)

# Inspect structure to avoid assuming an incorrect JSON format
print("JSON type:", type(data).__name__)
if isinstance(data, dict):
    print("Top-level keys:", list(data.keys()))
    for key, value in data.items():
        if isinstance(value, list):
            print(f"List found: {key}, length={len(value)}")

# 2. Find the user-level records
if isinstance(data, list):
    records = data
elif isinstance(data, dict):
    possible_keys = ["results", "pilot_results", "records", "tasks"]
    records = next(
        (data[k] for k in possible_keys
         if isinstance(data.get(k), list)),
        None
    )
    if records is None:
        raise ValueError(
            "Cannot locate user records. Please share the JSON structure."
        )
else:
    raise ValueError("Unexpected JSON structure")

df = pd.DataFrame(records)
print("\nAvailable columns:", df.columns.tolist())

# 3. Identify actual outcome fields
hit_col = next(
    (c for c in ["hit_at_5", "hit@5", "hit5", "hit"]
     if c in df.columns),
    None
)
rank_col = next(
    (c for c in ["target_rank", "groundtruth_rank", "rank"]
     if c in df.columns),
    None
)

if hit_col is None and rank_col is None:
    raise ValueError(
        "No Hit@5 or target rank field found. Share printed columns."
    )

if hit_col is not None:
    df["hit"] = pd.to_numeric(df[hit_col], errors="coerce")
else:
    df["hit"] = (
        pd.to_numeric(df[rank_col], errors="coerce") <= 5
    ).astype(int)

assert len(df) == 10, "Expected exactly 10 pilot tasks"
assert df["hit"].notna().all(), "Some outcomes are missing"
assert df["hit"].isin([0, 1]).all(), "Invalid Hit@5 values"

hits = int(df["hit"].sum())
failures = len(df) - hits

print(f"\nValidated results: {hits} hits, {failures} failures")

# 4. Create publication-style figure
fig, axes = plt.subplots(
    1, 2, figsize=(9.5, 2.7),
    gridspec_kw={"width_ratios": [1.35, 1]}
)

hit_color = "#76A996"
fail_color = "#D7A081"

# Panel A: Individual users
x = np.arange(1, len(df) + 1)
colors = [
    hit_color if h == 1 else fail_color
    for h in df["hit"]
]

axes[0].bar(
    x, np.ones(len(df)),
    color=colors, width=0.78
)

for i, hit in enumerate(df["hit"]):
    axes[0].text(
        i + 1, 0.5,
        "H" if hit == 1 else "F",
        ha="center", va="center",
        fontsize=12, fontweight="bold",
        color="#26352F"
    )

axes[0].set_xticks(x)
axes[0].set_xticklabels([str(i) for i in x])
axes[0].set_yticks([])
axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("Pilot task")
axes[0].set_title(
    "A. User-level outcomes",
    fontweight="bold", fontsize=11
)
axes[0].spines[:].set_visible(False)

# Panel B: Aggregate outcomes
axes[1].barh(
    ["Pilot"],
    [hits],
    color=hit_color,
    height=0.42,
    label="Hit@5"
)
axes[1].barh(
    ["Pilot"],
    [failures],
    left=[hits],
    color=fail_color,
    height=0.42,
    label="Failure@5"
)

axes[1].text(
    hits / 2, 0,
    f"{hits} Hits\n({hits/10:.0%})",
    ha="center", va="center",
    fontsize=11, fontweight="bold"
)
axes[1].text(
    hits + failures / 2, 0,
    f"{failures} Failures\n({failures/10:.0%})",
    ha="center", va="center",
    fontsize=10, fontweight="bold"
)

axes[1].set_xlim(0, 10)
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_title(
    "B. Overall outcomes",
    fontweight="bold", fontsize=11
)
axes[1].spines[:].set_visible(False)

fig.suptitle(
    "Gemini Recommendation Pilot (n = 10)",
    fontsize=13, fontweight="bold"
)

plt.tight_layout()

plt.savefig(
    "/content/Figure4_Gemini_Pilot.png",
    dpi=300, bbox_inches="tight"
)
plt.savefig(
    "/content/Figure4_Gemini_Pilot.pdf",
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("/content/Figure4_Gemini_Pilot.png")
print("/content/Figure4_Gemini_Pilot.pdf")


In [ ]:

from pathlib import Path

folder = Path("/content")

expected_files = [
    "amazon_tvd_final.csv",
    "amazon_recommendation_tasks.json",
    "gemini_pilot_10.json",
    "recommendation_experiment_config.json",
    "Figure2_TVD_Validation.png",
    "Figure4_Gemini_Pilot.png",
]

print("=== ATTEMPT 2 FILE CHECK ===\n")

for filename in expected_files:
    path = folder / filename

    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"FOUND   {filename} ({size_kb:.1f} KB)")
    else:
        print(f"MISSING {filename}")

print("\n=== NOTEBOOK CHECK ===")
notebooks = list(folder.glob("*.ipynb"))

if notebooks:
    for notebook in notebooks:
        print("FOUND  ", notebook.name)
else:
    print("No .ipynb file found directly in /content.")
    print("This is normal for notebooks opened from Google Drive.")


In [ ]:

from pathlib import Path
import shutil

source = Path("/content")
destination = Path("/content/attempt2")

file_mapping = {
    "amazon_tvd_final.csv": "data",
    "amazon_recommendation_tasks.json": "data",
    "gemini_pilot_10.json": "results",
    "recommendation_experiment_config.json": "config",
    "Figure2_TVD_Validation.png": "figures",
    "Figure4_Gemini_Pilot.png": "figures",
}

print("=== PREPARING ATTEMPT 2 ===\n")

for filename, subfolder in file_mapping.items():
    src = source / filename
    dst_dir = destination / subfolder
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst = dst_dir / filename
    shutil.copy2(src, dst)

    print(f"OK  attempt2/{subfolder}/{filename}")

print("\n=== SUMMARY ===")
print("Files copied:", len(file_mapping))
print("Destination:", destination)


In [ ]:

from pathlib import Path
import shutil

source = Path("/content/attempt2_pilot.ipynb")
destination = Path("/content/attempt2/attempt2_pilot.ipynb")

if source.exists():
    shutil.copy2(source, destination)
    print("SUCCESS: Notebook added to attempt2/")
    print("File size:", round(destination.stat().st_size / 1024, 1), "KB")
else:
    print("Notebook not found.")
    print("Please upload attempt2_pilot.ipynb to Colab Files first.")
